# 05 — Evidence-Based Research Synthesis

**Scenario:** decide whether the fictional Atlas database migration is ready for production.  
**Question:** *What are the benefits, risks, and unresolved concerns of migrating to Atlas?*

This lab preserves Map-Reduce, Refine, and the Engineering-versus-QA conflict, but makes every intermediate evidence decision inspectable.

> **Central rule:** Research synthesis is evidence integration, not “retrieve many documents and summarize.”

![Map-Reduce and Refine patterns](assets/synthesis-patterns.svg)


## Learning objectives and success criteria

You will build this bounded workflow:

```text
question → evidence plan → focused views → typed evidence records
         → source-family/conflict/gap analysis → claim-evidence map
         → cited synthesis → evaluation
```

The output must cover cost, performance, security, and operational readiness; retain stable evidence IDs; expose known conflicts; distinguish repeated citations from independent evidence; qualify temporal changes; and report missing evidence.

The default mode is deterministic and credential-free. Set `RUN_LIVE_SYNTHESIS=1`, `OPENAI_API_KEY`, and optionally `SYNTHESIS_MODEL` to run schema-constrained extraction and reduction. Frozen artifacts are reproducibility fixtures—not claimed live-model results.


## 1. Setup and typed contracts

The model may propose claim content, but the application owns provenance. `ExtractedClaim` intentionally has no source or evidence ID. Trusted source metadata is attached only after extraction.


In [ ]:
from __future__ import annotations

import json
import math
import os
import re
import time
from collections import defaultdict
from typing import Literal

import matplotlib.pyplot as plt
import pandas as pd
from pydantic import BaseModel, ConfigDict, Field
from IPython.display import display, Markdown

RUN_LIVE = os.getenv("RUN_LIVE_SYNTHESIS", "0") == "1"
MODEL_NAME = os.getenv("SYNTHESIS_MODEL", "gpt-5-mini")
QUESTION = "What are the benefits, risks, and unresolved concerns of migrating to Atlas?"

ClaimType = Literal["benefit", "risk", "constraint", "observation", "recommendation"]
ConflictType = Literal["direct", "temporal", "scope", "definition", "unresolved"]
ResolutionStatus = Literal["resolved", "partially_resolved", "unresolved"]


class SourceDocument(BaseModel):
    model_config = ConfigDict(frozen=True)
    source_id: str
    document_id: str
    evidence_id: str
    title: str
    source_type: str
    authority: str
    date: str | None
    source_family: str
    version: str
    department: str
    topic: str
    content: str


class ExtractedClaim(BaseModel):
    claim: str
    claim_type: ClaimType
    scope: str | None = None
    supports_question: bool
    extraction_status: Literal["extracted", "irrelevant", "needs_review"]


class EvidenceRecord(BaseModel):
    model_config = ConfigDict(frozen=True)
    evidence_id: str
    source_id: str
    source_family: str
    claim: str
    claim_type: ClaimType
    date: str | None
    scope: str | None
    authority: str
    supports_question: bool
    extraction_status: str
    topic: str


class ConflictRecord(BaseModel):
    topic: str
    evidence_ids: list[str]
    conflict_type: ConflictType
    explanation: str
    resolution_status: ResolutionStatus


class ClaimEvidence(BaseModel):
    topic: str
    claim: str
    supporting_evidence: list[str]
    contradicting_evidence: list[str] = Field(default_factory=list)
    source_families: list[str]
    status: str
    material: bool = True


class SynthesisState(BaseModel):
    records: list[EvidenceRecord] = Field(default_factory=list)
    conflicts: list[ConflictRecord] = Field(default_factory=list)
    gaps: list[str] = Field(default_factory=list)


class SynthesisOutput(BaseModel):
    report: str
    cited_aliases: list[str]
    disclosed_gaps: list[str]


print({"mode": "live" if RUN_LIVE else "offline-frozen", "model": MODEL_NAME if RUN_LIVE else None})


## 2. Build a heterogeneous Atlas evidence corpus

The 28 synthetic evidence items deliberately mix measurements, audits, forecasts, vendor claims, derivative summaries, current and historical reports, and irrelevant documents. Authority labels are a fictional teaching taxonomy, not universal truth scores.


In [ ]:
RAW_SOURCES = [
    ("fin-benchmark-2026", "fin-benchmark", "fin-benchmark-2026#cost", "Finance Compute Benchmark", "benchmark", "primary_internal_measurement", "2026-05-10", "finance-benchmark", "2", "finance", "cost", "Measured Atlas compute spend was 30% below the legacy platform for the benchmark workload."),
    ("exec-atlas-memo", "exec-atlas-memo", "exec-atlas-memo#cost", "Atlas Executive Memo", "internal_memo", "secondary_summary", "2026-05-18", "finance-benchmark", "1", "finance", "cost", "The Finance benchmark showed that Atlas reduced compute cost by 30%."),
    ("migration-steerco-deck", "migration-deck", "migration-steerco-deck#cost", "Migration Steering Deck", "slide_deck", "secondary_summary", "2026-05-22", "finance-benchmark", "3", "architecture", "cost", "Atlas can cut compute cost by 30%, according to the Finance benchmark."),
    ("fin-tco-2026", "fin-tco", "fin-tco-2026#total-cost", "Atlas Total Cost Model", "forecast", "primary_internal_measurement", "2026-06-02", "finance-tco", "1", "finance", "cost", "Including licenses and migration labor, projected monthly total cost falls by 12%, not 30%."),
    ("procurement-renewal", "procurement-renewal", "procurement-renewal#license", "Atlas License Renewal", "contract_review", "official_policy", "2026-06-20", "atlas-commercial", "1", "finance", "cost", "The second-year Atlas license price increases by 8% unless usage commitments are met."),
    ("eng-normal-load-2026", "eng-latency", "eng-normal-load-2026#latency", "Engineering Normal-Load Test", "internal_test", "primary_internal_measurement", "2026-05-03", "engineering-benchmark", "2", "engineering", "latency", "Atlas P99 latency was 45 ms at normal weekday traffic and Engineering marked the service production-ready."),
    ("qa-stress-may-2026", "qa-load-test", "qa-stress-may-2026#latency", "QA May Stress Test", "internal_test", "primary_internal_measurement", "2026-05-12", "qa-load-test", "1", "qa", "latency", "Atlas P99 latency reached 800 ms at 4x traffic; QA advised against production release."),
    ("qa-stress-july-2026", "qa-load-test", "qa-stress-july-2026#latency", "QA July Optimized Stress Test", "internal_test", "primary_internal_measurement", "2026-07-14", "qa-load-test", "2", "qa", "latency", "After index tuning, Atlas P99 latency was 120 ms at 4x traffic, within the revised 150 ms stress target."),
    ("qa-stress-april-2026", "qa-load-test", "qa-stress-april-2026#latency", "QA April Baseline", "historical_report", "primary_internal_measurement", "2026-04-18", "qa-load-test", "0", "qa", "latency", "Before tuning, Atlas P99 latency reached 950 ms at 4x traffic."),
    ("vendor-perf-brief", "vendor-perf", "vendor-perf-brief#latency", "Vendor Performance Brief", "vendor_claim", "vendor_claim", "2026-06-01", "vendor-performance", "4", "vendor", "latency", "Vendor synthetic tests report 40 ms P99 latency but do not disclose the customer workload mix."),
    ("eng-readiness-note", "eng-readiness", "eng-readiness-note#readiness", "Engineering Readiness Note", "internal_memo", "secondary_summary", "2026-06-05", "engineering-readiness", "1", "engineering", "readiness", "Engineering considers Atlas production-ready after the normal-load benchmark."),
    ("qa-release-gate", "qa-release-gate", "qa-release-gate#readiness", "QA Release Gate", "official_policy", "official_policy", "2026-06-07", "qa-governance", "1", "qa", "readiness", "QA will not approve production until rollback and 4x-load tests pass."),
    ("security-audit-internal", "security-audit", "security-audit-internal#controls", "Internal Security Audit", "audit", "primary_internal_measurement", "2026-06-10", "security-audit", "1", "security", "security", "No critical vulnerabilities were found; two medium findings require key-rotation and logging remediation."),
    ("security-audit-independent", "security-audit", "security-audit-independent#logging", "Independent Access Audit", "audit", "independent_audit", "2026-06-24", "independent-access-audit", "1", "security", "security", "The independent audit found incomplete privileged-query logging in the Atlas pilot."),
    ("compliance-residency", "compliance-residency", "compliance-residency#eu", "EU Data Residency Assessment", "compliance_assessment", "official_policy", "2026-06-26", "residency-policy", "1", "compliance", "compliance", "EU production approval requires evidence that backup replicas remain in approved regions."),
    ("ops-failover-drill", "ops-failover", "ops-failover-drill#rto", "Atlas Failover Drill", "incident_report", "primary_internal_measurement", "2026-06-15", "operations-drill", "1", "operations", "operations", "The failover drill recovered service in 22 minutes against a 15-minute objective."),
    ("ops-backup-test", "ops-backup", "ops-backup-test#restore", "Backup Restore Test", "internal_test", "primary_internal_measurement", "2026-06-19", "operations-backup", "1", "operations", "operations", "A sampled Atlas backup restored successfully in 8 minutes with no row-count mismatch."),
    ("ops-rollback-runbook", "ops-rollback", "ops-rollback-runbook#procedure", "Atlas Rollback Runbook", "runbook", "official_policy", "2026-07-02", "rollback-runbook", "2", "operations", "rollback", "Rollback uses a dual-write checkpoint and was rehearsed in 35 minutes; Incident Command owns authorization."),
    ("architecture-dual-write", "architecture-dual-write", "architecture-dual-write#complexity", "Dual-Write Architecture Review", "architecture_review", "primary_internal_measurement", "2026-05-28", "atlas-architecture", "3", "architecture", "migration", "Dual-write reduces cutover risk but adds consistency monitoring and reconciliation work."),
    ("migration-rehearsal", "migration-rehearsal", "migration-rehearsal#integrity", "Migration Rehearsal Report", "internal_test", "primary_internal_measurement", "2026-06-30", "migration-rehearsal", "2", "operations", "migration", "The rehearsal found a 0.02% row mismatch that was corrected by replaying the change log."),
    ("support-pilot", "support-pilot", "support-pilot#tickets", "Support Pilot Review", "production_observation", "primary_internal_measurement", "2026-07-03", "support-pilot", "1", "support", "operations", "The Atlas pilot reduced database-related support tickets by 18% over four weeks."),
    ("vendor-sla", "vendor-sla", "vendor-sla#availability", "Atlas Vendor SLA", "vendor_claim", "vendor_claim", "2026-05-01", "vendor-commercial", "5", "vendor", "readiness", "The vendor contract promises 99.99% service availability, subject to listed exclusions."),
    ("architecture-capacity", "architecture-capacity", "architecture-capacity#forecast", "Atlas Capacity Forecast", "forecast", "secondary_summary", "2026-07-08", "capacity-forecast", "1", "architecture", "readiness", "Forecasts show 18 months of headroom, but the forecast has not been validated with seasonal traffic."),
    ("ops-pilot-observation", "ops-pilot", "ops-pilot-observation#stability", "Operations Pilot Observation", "production_observation", "primary_internal_measurement", "2026-07-10", "operations-pilot", "1", "operations", "readiness", "The limited pilot ran for 72 hours without a severity-one incident."),
    ("cafeteria-budget", "cafeteria-budget", "cafeteria-budget#cost", "Cafeteria Budget", "internal_memo", "secondary_summary", "2026-06-01", "workplace-budget", "1", "finance", "cost", "The cafeteria coffee contract is 30% over budget."),
    ("hr-performance-cycle", "hr-performance", "hr-performance-cycle#latency", "Performance Review Calendar", "internal_memo", "official_policy", "2026-05-01", "hr-policy", "1", "hr", "latency", "Manager performance reviews must be submitted within 45 days."),
    ("physical-security-badges", "physical-security", "physical-security-badges#controls", "Office Badge Audit", "audit", "independent_audit", "2026-06-11", "physical-security", "1", "facilities", "security", "Three expired office badges remained active after employee departures."),
    ("office-relocation-plan", "office-relocation", "office-relocation-plan#operations", "Office Relocation Plan", "internal_memo", "secondary_summary", "2026-07-01", "facilities-plan", "1", "facilities", "operations", "The Vancouver office move is scheduled for September."),
]

CORPUS = [SourceDocument(
    source_id=s[0], document_id=s[1], evidence_id=s[2], title=s[3], source_type=s[4],
    authority=s[5], date=s[6], source_family=s[7], version=s[8], department=s[9],
    topic=s[10], content=s[11]
) for s in RAW_SOURCES]

assert 20 <= len(CORPUS) <= 35
assert len({d.evidence_id for d in CORPUS}) == len(CORPUS)
display(pd.DataFrame([d.model_dump(exclude={"content"}) for d in CORPUS]).head(10))
print(f"Corpus: {len(CORPUS)} evidence items, {len({d.source_family for d in CORPUS})} source families")


## 3. Create a bounded evidence plan and focused views

The plan is a coverage contract, not an unbounded agent. The initial views intentionally omit the rollback-specific document so the gap-filling step has real work to do. Irrelevant documents share surface terms with the question and test whether mapping can decline to extract a claim.


In [ ]:
RESEARCH_PLAN = {
    "benefits": ["cost", "performance", "operational_efficiency"],
    "risks": ["performance_under_load", "security", "migration_operations"],
    "decision_questions": ["production_readiness", "disputed_claims"],
    "required_topics": ["cost", "performance", "security", "operational_readiness", "rollback", "customer_communication"],
}

VIEW_TOPICS = {
    "cost": {"cost"},
    "performance": {"latency"},
    "security": {"security", "compliance"},
    "operational_readiness": {"readiness", "operations", "migration"},
}


def focused_view(name: str, corpus: list[SourceDocument]) -> list[SourceDocument]:
    allowed = VIEW_TOPICS[name]
    return [doc for doc in corpus if doc.topic in allowed]


views = {name: focused_view(name, CORPUS) for name in VIEW_TOPICS}
for name, docs in views.items():
    print(f"{name:24s} {len(docs):2d} items: {[d.source_id for d in docs]}")

selected_by_id = {doc.evidence_id: doc for docs in views.values() for doc in docs}
SELECTED = list(selected_by_id.values())
assert all(doc.topic != "rollback" for doc in SELECTED)
print(f"Unique initial evidence candidates: {len(SELECTED)} / {len(CORPUS)}")


## 4. Map: structured extraction with a live/frozen provider pattern

`FROZEN_EXTRACTIONS` is a committed reproducibility fixture. In live mode, `ChatOpenAI.with_structured_output()` produces only `ExtractedClaim`; `attach_provenance()` adds trusted IDs and metadata afterward. A schema constrains shape, not truth, so every record is validated against the corpus.


In [ ]:
IRRELEVANT_IDS = {
    "cafeteria-budget#cost", "hr-performance-cycle#latency",
    "physical-security-badges#controls", "office-relocation-plan#operations",
}

CLAIM_OVERRIDES = {
    "fin-benchmark-2026#cost": ("Atlas reduced compute-only cost by 30% in the Finance benchmark.", "benefit", "benchmark workload"),
    "exec-atlas-memo#cost": ("The executive memo repeats the benchmark's 30% compute-cost result.", "benefit", "derivative summary"),
    "migration-steerco-deck#cost": ("The steering deck repeats the benchmark's 30% compute-cost result.", "benefit", "derivative summary"),
    "fin-tco-2026#total-cost": ("Projected total monthly cost falls by 12% when licenses and migration labor are included.", "benefit", "total monthly cost"),
    "procurement-renewal#license": ("Second-year license cost rises by 8% if commitments are not met.", "risk", "commercial terms"),
    "eng-normal-load-2026#latency": ("Atlas P99 latency was 45 ms under normal weekday traffic.", "benefit", "normal load"),
    "qa-stress-may-2026#latency": ("Atlas P99 latency was 800 ms at 4x traffic in May 2026.", "risk", "4x stress load"),
    "qa-stress-july-2026#latency": ("After tuning, Atlas P99 latency was 120 ms at 4x traffic in July 2026.", "observation", "4x stress load after tuning"),
    "qa-stress-april-2026#latency": ("Atlas P99 latency was 950 ms at 4x traffic in April 2026.", "risk", "4x stress load before tuning"),
    "vendor-perf-brief#latency": ("The vendor reports 40 ms P99 on an undisclosed synthetic workload.", "observation", "vendor synthetic test"),
    "eng-readiness-note#readiness": ("Engineering considers Atlas production-ready.", "recommendation", "engineering release opinion"),
    "qa-release-gate#readiness": ("QA withholds production approval until rollback and 4x-load tests pass.", "constraint", "QA release gate"),
    "security-audit-internal#controls": ("No critical findings were found, but key rotation and logging remediation remain.", "risk", "internal security audit"),
    "security-audit-independent#logging": ("Privileged-query logging was incomplete in the pilot.", "risk", "independent access audit"),
    "compliance-residency#eu": ("EU approval requires evidence that backup replicas remain in approved regions.", "constraint", "EU workloads"),
    "ops-failover-drill#rto": ("Failover took 22 minutes against a 15-minute objective.", "risk", "failover drill"),
    "ops-backup-test#restore": ("A sampled backup restored in 8 minutes without row-count mismatch.", "benefit", "sample restore"),
    "ops-rollback-runbook#procedure": ("The rehearsed rollback uses a dual-write checkpoint and took 35 minutes.", "observation", "rollback rehearsal"),
    "architecture-dual-write#complexity": ("Dual-write reduces cutover risk but adds reconciliation work.", "risk", "migration architecture"),
    "migration-rehearsal#integrity": ("The rehearsal found a 0.02% row mismatch corrected through log replay.", "risk", "migration rehearsal"),
    "support-pilot#tickets": ("Database-related support tickets fell by 18% during the four-week pilot.", "benefit", "limited support pilot"),
    "vendor-sla#availability": ("The vendor promises 99.99% availability subject to exclusions.", "observation", "contractual SLA"),
    "architecture-capacity#forecast": ("The forecast shows 18 months of headroom without seasonal validation.", "observation", "unvalidated forecast"),
    "ops-pilot-observation#stability": ("The limited pilot ran 72 hours without a severity-one incident.", "observation", "limited pilot"),
}


def frozen_extraction(doc: SourceDocument) -> ExtractedClaim:
    if doc.evidence_id in IRRELEVANT_IDS:
        return ExtractedClaim(
            claim="No Atlas migration evidence in this source.", claim_type="observation",
            scope=None, supports_question=False, extraction_status="irrelevant"
        )
    claim, claim_type, scope = CLAIM_OVERRIDES[doc.evidence_id]
    return ExtractedClaim(
        claim=claim, claim_type=claim_type, scope=scope,
        supports_question=True, extraction_status="extracted"
    )


def live_extraction(doc: SourceDocument) -> ExtractedClaim:
    if not os.getenv("OPENAI_API_KEY"):
        raise RuntimeError("RUN_LIVE_SYNTHESIS=1 requires OPENAI_API_KEY")
    from langchain_openai import ChatOpenAI
    extractor = ChatOpenAI(model=MODEL_NAME, temperature=0).with_structured_output(
        ExtractedClaim, method="json_schema"
    )
    return extractor.invoke(
        "Extract at most one material claim for the Atlas migration question. "
        "Mark unrelated content supports_question=false. Do not create source IDs.\n\n"
        f"Question: {QUESTION}\nSource content: {doc.content}"
    )


def attach_provenance(doc: SourceDocument, extracted: ExtractedClaim) -> EvidenceRecord:
    return EvidenceRecord(
        evidence_id=doc.evidence_id, source_id=doc.source_id,
        source_family=doc.source_family, claim=extracted.claim,
        claim_type=extracted.claim_type, date=doc.date, scope=extracted.scope,
        authority=doc.authority, supports_question=extracted.supports_question,
        extraction_status=extracted.extraction_status, topic=doc.topic,
    )


def validate_evidence(records: list[EvidenceRecord], corpus: list[SourceDocument]) -> None:
    by_evidence = {doc.evidence_id: doc for doc in corpus}
    if len({r.evidence_id for r in records}) != len(records):
        raise ValueError("Duplicate evidence IDs")
    for record in records:
        source = by_evidence.get(record.evidence_id)
        if source is None:
            raise ValueError(f"Unknown evidence ID: {record.evidence_id}")
        trusted = (source.source_id, source.source_family, source.date, source.authority)
        observed = (record.source_id, record.source_family, record.date, record.authority)
        if trusted != observed:
            raise ValueError(f"Provenance mismatch for {record.evidence_id}")


map_started = time.perf_counter()
extractor = live_extraction if RUN_LIVE else frozen_extraction
MAPPED = [attach_provenance(doc, extractor(doc)) for doc in SELECTED]
MAP_LATENCY_MS = (time.perf_counter() - map_started) * 1000
validate_evidence(MAPPED, CORPUS)

mapped_df = pd.DataFrame([r.model_dump() for r in MAPPED])
display(mapped_df[["evidence_id", "source_id", "claim", "claim_type", "date", "scope", "authority", "supports_question"]])
print(f"Discarded as irrelevant: {(~mapped_df.supports_question).sum()} of {len(mapped_df)} mapped candidates")


### Failure injection: provenance cannot be model-controlled

The next cell tampers with a source ID. Structural validation catches the failure before the record can enter synthesis.


In [ ]:
tampered = MAPPED[0].model_copy(update={"source_id": "invented-source"})
try:
    validate_evidence([tampered], CORPUS)
    raise AssertionError("Tampered provenance was not rejected")
except ValueError as exc:
    print("Expected fail-closed result:", exc)


## 5. Source-family deduplication: three citations are not three confirmations

The Finance benchmark, executive memo, and steering deck are different documents but one evidence lineage. Source-family counting makes citation laundering visible without automatically discarding derivative sources.


In [ ]:
RELEVANT = [r for r in MAPPED if r.supports_question]


def independent_source_count(records: list[EvidenceRecord]) -> int:
    return len({record.source_family for record in records})


cost_30 = [r for r in RELEVANT if "30%" in r.claim]
independence = {
    "citation_count": len(cost_30),
    "unique_source_count": len({r.source_id for r in cost_30}),
    "independent_source_family_count": independent_source_count(cost_30),
}
print(independence)
assert independence == {"citation_count": 3, "unique_source_count": 3, "independent_source_family_count": 1}


## 6. Group claims and detect conflicts before prose

The detector is intentionally explicit for this labelled teaching corpus. It distinguishes scope, time, definition, and direct disagreement before a reducer can smooth them into false consensus.

![Conflict types and handling](assets/conflict-handling.svg)


In [ ]:
def group_claims(records: list[EvidenceRecord]) -> dict[str, list[EvidenceRecord]]:
    grouped: dict[str, list[EvidenceRecord]] = defaultdict(list)
    for record in records:
        if record.supports_question:
            grouped[record.topic].append(record)
    return dict(grouped)


def analyze_conflicts(records: list[EvidenceRecord]) -> list[ConflictRecord]:
    available = {r.evidence_id for r in records}
    candidates = [
        ConflictRecord(topic="latency_by_load", evidence_ids=["eng-normal-load-2026#latency", "qa-stress-may-2026#latency"], conflict_type="scope", explanation="45 ms normal-load and 800 ms stress-load results describe different traffic conditions.", resolution_status="partially_resolved"),
        ConflictRecord(topic="latency_over_time", evidence_ids=["qa-stress-may-2026#latency", "qa-stress-july-2026#latency"], conflict_type="temporal", explanation="The July 120 ms result follows tuning; it updates but does not erase the May 800 ms result.", resolution_status="resolved"),
        ConflictRecord(topic="production_readiness", evidence_ids=["eng-readiness-note#readiness", "qa-release-gate#readiness"], conflict_type="direct", explanation="Engineering recommends release while the QA gate withholds approval.", resolution_status="unresolved"),
        ConflictRecord(topic="cost_reduction", evidence_ids=["fin-benchmark-2026#cost", "fin-tco-2026#total-cost"], conflict_type="definition", explanation="Thirty percent is compute-only; twelve percent includes licenses and migration labor.", resolution_status="partially_resolved"),
        ConflictRecord(topic="availability_readiness", evidence_ids=["vendor-sla#availability", "ops-failover-drill#rto"], conflict_type="unresolved", explanation="A contractual availability promise does not resolve whether observed recovery performance can meet production objectives.", resolution_status="unresolved"),
    ]
    return [c for c in candidates if set(c.evidence_ids).issubset(available)]


GROUPS = group_claims(RELEVANT)
CONFLICTS = analyze_conflicts(RELEVANT)
for topic, records in GROUPS.items():
    print(f"{topic:12s}: {len(records)} claims")
display(pd.DataFrame([c.model_dump() for c in CONFLICTS]))


## 7. Gap analysis and one bounded gap-filling round

Initial views have no rollback evidence and the corpus has no customer-communication plan. One targeted local search may fill rollback; the remaining gap must survive into the report.


In [ ]:
TOPIC_TO_PLAN = {
    "cost": "cost", "latency": "performance", "security": "security",
    "compliance": "security", "readiness": "operational_readiness",
    "operations": "operational_readiness", "migration": "operational_readiness",
    "rollback": "rollback",
}


def covered_plan_topics(records: list[EvidenceRecord]) -> set[str]:
    return {TOPIC_TO_PLAN[r.topic] for r in records if r.supports_question and r.topic in TOPIC_TO_PLAN}


def evidence_gaps(records: list[EvidenceRecord]) -> list[str]:
    covered = covered_plan_topics(records)
    return [topic for topic in RESEARCH_PLAN["required_topics"] if topic not in covered]


initial_gaps = evidence_gaps(RELEVANT)
print("Initial gaps:", initial_gaps)


def targeted_local_search(query: str, corpus: list[SourceDocument], limit: int = 3) -> list[SourceDocument]:
    terms = {term.lower() for term in re.findall(r"[a-z]+", query) if len(term) > 3}
    scored = []
    for doc in corpus:
        haystack = f"{doc.title} {doc.topic} {doc.content}".lower()
        score = sum(term in haystack for term in terms)
        if score:
            scored.append((score, doc.date or "", doc))
    return [doc for _, _, doc in sorted(scored, key=lambda item: (item[0], item[1]), reverse=True)[:limit]]


gap_candidates = targeted_local_search("Atlas rollback procedure strategy", CORPUS, limit=2)
print("Bounded gap-search results:", [d.evidence_id for d in gap_candidates])
gap_records = [attach_provenance(doc, extractor(doc)) for doc in gap_candidates if doc.evidence_id not in {r.evidence_id for r in MAPPED}]
validate_evidence(gap_records, CORPUS)
FINAL_RECORDS = RELEVANT + [r for r in gap_records if r.supports_question]
FINAL_GAPS = evidence_gaps(FINAL_RECORDS)
print("Gaps after one round:", FINAL_GAPS)
assert "rollback" not in FINAL_GAPS
assert "customer_communication" in FINAL_GAPS


## 8. Build the claim-evidence map

This is the central pre-generation artifact. It preserves supporting and contradicting evidence, source families, and interpretation status before style enters the workflow.


In [ ]:
by_id = {r.evidence_id: r for r in FINAL_RECORDS}


def claim_entry(topic: str, claim: str, supporting: list[str], contradicting: list[str], status: str) -> ClaimEvidence:
    all_ids = supporting + contradicting
    missing = [eid for eid in all_ids if eid not in by_id]
    if missing:
        raise ValueError(f"Claim map references unknown evidence: {missing}")
    return ClaimEvidence(
        topic=topic, claim=claim, supporting_evidence=supporting,
        contradicting_evidence=contradicting,
        source_families=sorted({by_id[eid].source_family for eid in all_ids}),
        status=status,
    )


CLAIM_MAP = [
    claim_entry("cost", "Atlas may reduce cost, but the measured reduction depends on the cost boundary.", ["fin-benchmark-2026#cost", "exec-atlas-memo#cost", "migration-steerco-deck#cost", "fin-tco-2026#total-cost"], ["procurement-renewal#license"], "definition-qualified"),
    claim_entry("performance", "Atlas latency improved after tuning but remains workload- and date-dependent.", ["eng-normal-load-2026#latency", "qa-stress-may-2026#latency", "qa-stress-july-2026#latency"], [], "scope-and-time-qualified"),
    claim_entry("security", "Security work remains despite no critical internal-audit finding.", ["security-audit-internal#controls", "security-audit-independent#logging", "compliance-residency#eu"], [], "open-controls"),
    claim_entry("operational_readiness", "Operational evidence is mixed and the production-release decision remains disputed.", ["ops-failover-drill#rto", "ops-backup-test#restore", "ops-pilot-observation#stability", "migration-rehearsal#integrity"], ["eng-readiness-note#readiness", "qa-release-gate#readiness"], "unresolved"),
    claim_entry("operational_efficiency", "The limited support pilot reported fewer database-related tickets.", ["support-pilot#tickets"], [], "single-independent-source"),
    claim_entry("rollback", "A rollback procedure exists and was rehearsed, but took 35 minutes.", ["ops-rollback-runbook#procedure"], [], "single-independent-source"),
]

display(pd.DataFrame([c.model_dump() for c in CLAIM_MAP]))


## 9. Application-assigned evidence aliases and structured Reduce

The generator sees aliases such as `[E4]`; the application resolves each alias back to a stable `evidence_id` and source. The reducer receives the question, claim map, conflict records, and gaps—not raw unstructured documents.


In [ ]:
referenced_ids = sorted({eid for c in CLAIM_MAP for eid in c.supporting_evidence + c.contradicting_evidence})
ALIAS_TO_EVIDENCE = {f"E{i}": eid for i, eid in enumerate(referenced_ids, start=1)}
EVIDENCE_TO_ALIAS = {eid: alias for alias, eid in ALIAS_TO_EVIDENCE.items()}


def cite(*evidence_ids: str) -> str:
    return " ".join(f"[{EVIDENCE_TO_ALIAS[eid]}]" for eid in evidence_ids)


display(pd.DataFrame([
    {"alias": alias, "evidence_id": eid, "source_id": by_id[eid].source_id, "source_family": by_id[eid].source_family}
    for alias, eid in ALIAS_TO_EVIDENCE.items()
]))


In [ ]:
FROZEN_REPORT = f"""## Atlas migration evidence synthesis

**Cost.** The Finance benchmark measured a 30% compute-only reduction {cite('fin-benchmark-2026#cost')}; two other documents repeat that same source family {cite('exec-atlas-memo#cost', 'migration-steerco-deck#cost')}. A broader model projects only 12% total monthly savings once licenses and migration labor are included {cite('fin-tco-2026#total-cost')}, and renewal pricing can rise {cite('procurement-renewal#license')}.

**Performance.** Engineering measured 45 ms P99 at normal traffic {cite('eng-normal-load-2026#latency')}. QA measured 800 ms at 4x traffic in May 2026 {cite('qa-stress-may-2026#latency')}, improving to 120 ms after tuning in July 2026 {cite('qa-stress-july-2026#latency')}. These are scope and temporal differences, not one context-free latency value.

**Security and compliance.** The internal audit found no critical issue but retained medium remediation {cite('security-audit-internal#controls')}; an independent audit found incomplete privileged-query logging {cite('security-audit-independent#logging')}; and EU residency evidence remains required {cite('compliance-residency#eu')}.

**Operational readiness.** Backup restore and a limited pilot were encouraging {cite('ops-backup-test#restore', 'ops-pilot-observation#stability')}, while failover missed its objective and a migration rehearsal found a correctable mismatch {cite('ops-failover-drill#rto', 'migration-rehearsal#integrity')}. Engineering calls Atlas ready, but QA still withholds release approval {cite('eng-readiness-note#readiness', 'qa-release-gate#readiness')}. A rollback runbook exists and was rehearsed in 35 minutes {cite('ops-rollback-runbook#procedure')}.

**Other evidence and gaps.** A limited support pilot reported 18% fewer database-related tickets {cite('support-pilot#tickets')}. No source in the available evidence describes the customer communication plan. Production readiness therefore remains unresolved pending the QA gate, security/compliance evidence, and operational objectives."""


def live_reduce() -> SynthesisOutput:
    if not os.getenv("OPENAI_API_KEY"):
        raise RuntimeError("RUN_LIVE_SYNTHESIS=1 requires OPENAI_API_KEY")
    from langchain_openai import ChatOpenAI
    reducer = ChatOpenAI(model=MODEL_NAME, temperature=0).with_structured_output(
        SynthesisOutput, method="json_schema"
    )
    payload = {
        "question": QUESTION,
        "claim_evidence_map": [c.model_dump() for c in CLAIM_MAP],
        "conflicts": [c.model_dump() for c in analyze_conflicts(FINAL_RECORDS)],
        "gaps": FINAL_GAPS,
        "allowed_aliases": ALIAS_TO_EVIDENCE,
    }
    return reducer.invoke(
        "Write a concise decision synthesis using only the supplied claims. "
        "Cite only allowed aliases, preserve dates/scopes/conflicts, and disclose every gap.\n\n"
        + json.dumps(payload, indent=2)
    )


reduce_started = time.perf_counter()
if RUN_LIVE:
    REDUCED = live_reduce()
else:
    REDUCED = SynthesisOutput(
        report=FROZEN_REPORT,
        cited_aliases=sorted(set(re.findall(r"\[(E\d+)\]", FROZEN_REPORT))),
        disclosed_gaps=FINAL_GAPS,
    )
REDUCE_LATENCY_MS = (time.perf_counter() - reduce_started) * 1000
display(Markdown(REDUCED.report))


## 10. Implement structured Refine and expose order sensitivity

Each evidence item updates typed state. With unlimited retention, deterministic merging preserves the same evidence regardless of order. Under a deliberately bounded state capacity—standing in for compression or context limits—recency-based eviction makes order effects visible. This is a teaching failure mode, not a recommended retention policy.


In [ ]:
EXPECTED_CONFLICTS = {"latency_by_load", "latency_over_time", "production_readiness", "cost_reduction", "availability_readiness"}


def update_state(state: SynthesisState, record: EvidenceRecord, capacity: int | None = None) -> SynthesisState:
    records = [r for r in state.records if r.evidence_id != record.evidence_id]
    if record.supports_question:
        records.append(record)
    if capacity is not None and len(records) > capacity:
        records = records[-capacity:]  # explicit recency policy for the experiment
    return SynthesisState(records=records, conflicts=analyze_conflicts(records), gaps=evidence_gaps(records))


def run_refine(order: list[EvidenceRecord], capacity: int | None = None) -> tuple[SynthesisState, float]:
    state = SynthesisState(gaps=RESEARCH_PLAN["required_topics"])
    started = time.perf_counter()
    for record in order:
        state = update_state(state, record, capacity=capacity)
    return state, (time.perf_counter() - started) * 1000


department_order_a = {"engineering": 0, "qa": 1, "finance": 2, "security": 3, "compliance": 4, "operations": 5, "architecture": 6, "support": 7, "vendor": 8}
department_order_b = {"qa": 0, "finance": 1, "security": 2, "compliance": 3, "operations": 4, "architecture": 5, "support": 6, "vendor": 7, "engineering": 8}
source_department = {d.evidence_id: d.department for d in CORPUS}
order_a = sorted(FINAL_RECORDS, key=lambda r: (department_order_a.get(source_department[r.evidence_id], 99), r.date or ""))
order_b = sorted(FINAL_RECORDS, key=lambda r: (department_order_b.get(source_department[r.evidence_id], 99), r.date or ""))

full_a, _ = run_refine(order_a)
full_b, _ = run_refine(order_b)
assert {r.evidence_id for r in full_a.records} == {r.evidence_id for r in full_b.records}

bounded_a, refine_a_ms = run_refine(order_a, capacity=12)
bounded_b, refine_b_ms = run_refine(order_b, capacity=12)


def state_result(name: str, state: SynthesisState) -> dict:
    ids = {r.evidence_id for r in state.records}
    return {
        "order": name,
        "claims_retained": len(state.records),
        "conflicts_retained": len(state.conflicts),
        "evidence_ids_retained": len(ids),
        "conflict_coverage": len({c.topic for c in state.conflicts} & EXPECTED_CONFLICTS) / len(EXPECTED_CONFLICTS),
        "gaps": ", ".join(state.gaps),
        "final_state_summary": "; ".join(sorted({r.topic for r in state.records})),
    }


refine_comparison = pd.DataFrame([state_result("engineering-first", bounded_a), state_result("qa-first", bounded_b)])
display(refine_comparison)
print("Only in engineering-first bounded state:", sorted({r.evidence_id for r in bounded_a.records} - {r.evidence_id for r in bounded_b.records}))
print("Only in QA-first bounded state:", sorted({r.evidence_id for r in bounded_b.records} - {r.evidence_id for r in bounded_a.records}))


## 11. Evaluate support, citations, conflicts, independence, time, and gaps

These deterministic checks use labels built into the synthetic teaching corpus. They do not replace semantic entailment review or domain-expert judgment on real evidence.


In [ ]:
FINAL_CONFLICTS = analyze_conflicts(FINAL_RECORDS)


def evaluate_synthesis(report: str, claim_map: list[ClaimEvidence]) -> dict[str, float | int | bool]:
    known_ids = set(by_id)
    cited_aliases = set(re.findall(r"\[(E\d+)\]", report))
    cited_ids = {ALIAS_TO_EVIDENCE[a] for a in cited_aliases if a in ALIAS_TO_EVIDENCE}
    invalid_aliases = cited_aliases - set(ALIAS_TO_EVIDENCE)
    material = [claim for claim in claim_map if claim.material]
    claim_support = sum(
        bool(claim.supporting_evidence) and set(claim.supporting_evidence).issubset(known_ids)
        for claim in material
    ) / len(material)
    citation_completeness = sum(
        bool(set(claim.supporting_evidence) & cited_ids) for claim in material
    ) / len(material)
    mapped_topics = {claim.topic for claim in material}
    required_map_topics = {"cost", "performance", "security", "operational_readiness", "rollback"}
    all_sources = {by_id[eid].source_id for claim in material for eid in claim.supporting_evidence}
    all_families = {by_id[eid].source_family for claim in material for eid in claim.supporting_evidence}
    conflict_topics = {conflict.topic for conflict in FINAL_CONFLICTS}
    return {
        "claim_support": claim_support,
        "citation_validity": float(not invalid_aliases),
        "citation_completeness": citation_completeness,
        "required_topic_coverage": len(mapped_topics & required_map_topics) / len(required_map_topics),
        "conflict_coverage": len(conflict_topics & EXPECTED_CONFLICTS) / len(EXPECTED_CONFLICTS),
        "unique_source_count": len(all_sources),
        "unique_source_family_count": len(all_families),
        "source_family_diversity": len(all_families) / len(all_sources),
        "duplicate_source_rate": 1 - len(all_families) / len(all_sources),
        "temporal_qualification": float("May 2026" in report and "July 2026" in report),
        "evidence_gap_reporting": float(all(gap.replace("_", " ") in report.lower() for gap in FINAL_GAPS)),
    }


EVALUATION = evaluate_synthesis(REDUCED.report, CLAIM_MAP)
display(pd.DataFrame([EVALUATION]).T.rename(columns={0: "value"}))
assert EVALUATION["claim_support"] == 1.0
assert EVALUATION["citation_validity"] == 1.0
assert EVALUATION["citation_completeness"] == 1.0
assert EVALUATION["conflict_coverage"] == 1.0
assert EVALUATION["temporal_qualification"] == 1.0
assert EVALUATION["evidence_gap_reporting"] == 1.0


### Failure injection: smooth prose that omits a known conflict must fail

Citation validity cannot detect a missing disagreement. The release gate therefore checks expected conflict coverage independently.


In [ ]:
omitted_conflicts = [c for c in FINAL_CONFLICTS if c.topic != "production_readiness"]
bad_conflict_coverage = len({c.topic for c in omitted_conflicts} & EXPECTED_CONFLICTS) / len(EXPECTED_CONFLICTS)
print({"known_conflict_coverage": bad_conflict_coverage, "release_allowed": bad_conflict_coverage == 1.0})
assert bad_conflict_coverage < 1.0


## 12. Compare Map-Reduce and Refine using measured teaching artifacts

Offline mode has zero external model calls. The table reports logical strategy calls, observed local execution time, actual input characters, and an explicitly approximate token count (`ceil(characters / 4)`). It is not a network-latency or provider-cost benchmark.


In [ ]:
map_input_chars = sum(len(d.content) + len(QUESTION) for d in SELECTED)
reduce_input_chars = len(json.dumps({
    "question": QUESTION,
    "claims": [c.model_dump() for c in CLAIM_MAP],
    "conflicts": [c.model_dump() for c in FINAL_CONFLICTS],
    "gaps": FINAL_GAPS,
}))
refine_input_chars = sum(len(r.claim) for r in FINAL_RECORDS)

strategy_metrics = pd.DataFrame([
    {
        "strategy": "map-reduce",
        "map_calls": len(SELECTED), "reduce_calls": 1, "refine_calls": 0,
        "external_model_calls": len(SELECTED) + 1 if RUN_LIVE else 0,
        "input_characters": map_input_chars + reduce_input_chars,
        "estimated_input_tokens_offline": math.ceil((map_input_chars + reduce_input_chars) / 4),
        "measured_local_latency_ms": MAP_LATENCY_MS + REDUCE_LATENCY_MS,
        "claim_coverage": EVALUATION["required_topic_coverage"],
        "conflict_coverage": EVALUATION["conflict_coverage"],
        "citation_completeness": EVALUATION["citation_completeness"],
        "order_sensitive_under_cap": False,
    },
    {
        "strategy": "structured-refine (bounded)",
        "map_calls": 0, "reduce_calls": 0, "refine_calls": len(FINAL_RECORDS),
        "external_model_calls": len(FINAL_RECORDS) if RUN_LIVE else 0,
        "input_characters": refine_input_chars,
        "estimated_input_tokens_offline": math.ceil(refine_input_chars / 4),
        "measured_local_latency_ms": (refine_a_ms + refine_b_ms) / 2,
        "claim_coverage": len(covered_plan_topics(bounded_a.records) - {"customer_communication"}) / 5,
        "conflict_coverage": len({c.topic for c in bounded_a.conflicts} & EXPECTED_CONFLICTS) / len(EXPECTED_CONFLICTS),
        "citation_completeness": sum(bool({r.evidence_id for r in bounded_a.records} & set(claim.supporting_evidence)) for claim in CLAIM_MAP) / len(CLAIM_MAP),
        "evidence_retention": len(bounded_a.records) / len(FINAL_RECORDS),
        "order_sensitive_under_cap": {r.evidence_id for r in bounded_a.records} != {r.evidence_id for r in bounded_b.records},
    },
])
display(strategy_metrics)

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
strategy_metrics.set_index("strategy")[["claim_coverage", "conflict_coverage", "citation_completeness"]].plot.bar(ax=axes[0], ylim=(0, 1.05), title="Quality checks")
strategy_metrics.set_index("strategy")[["map_calls", "reduce_calls", "refine_calls"]].plot.bar(stacked=True, ax=axes[1], title="Logical call structure")
axes[0].set_ylabel("fraction")
axes[1].set_ylabel("calls")
plt.tight_layout()
plt.show()


## 13. What the experiment means

- Map-Reduce preserves global comparison when the reducer receives an explicit evidence structure; parallel speed is constrained by rate limits, batching, retries, model latency, and document length.
- Refine introduces sequential dependencies. With complete typed state it can be order-invariant for evidence retention; with compression or a capacity limit, order changes which evidence survives.
- Citation volume overstates independence when derivative sources share a family.
- A polished answer can pass citation validity and still fail conflict coverage or gap reporting.
- The local timings measure Python teaching code only. In live mode, record provider-reported usage and per-stage network latency rather than extrapolating from this fixture.


## 14. Production upgrade checklist

| Teaching lab | Production control |
|---|---|
| In-memory synthetic sources | Immutable authorized snapshots, hashes, lineage, versions |
| Keyword/metadata views | Evaluated retriever with source access policy |
| Frozen or optional extraction | Versioned prompts/models, retries, quarantine, semantic review |
| Explicit Python conflict rules | Domain rules plus calibrated model/human review |
| One bounded gap round | Time/cost/source budgets and escalation policy |
| Local state | Durable evidence, conflict, gap, and reviewer records |
| Character-based token estimate | Provider usage, cost, queue, retry, and latency traces |
| Deterministic teaching labels | Versioned representative evaluation set and release gates |

Security belongs before synthesis: authorize source eligibility before content reaches extraction, logs, caches, traces, or the model. Do not expose confidential evidence merely because the final prose is access-controlled.


## 15. Exercises

1. Add a derivative source and verify that citation count rises but independent family count does not.
2. Change the bounded Refine capacity and graph conflict coverage versus retained records.
3. Inject an unknown evidence ID into a claim map and make validation fail closed.
4. Add a material uncited claim and extend the completeness gate.
5. Add a second gap-filling round, then define a stopping policy with time and cost limits.
6. Replace recency eviction with an authority/diversity-aware retention policy and compare both orders.
7. Run live extraction into a new artifact; do not overwrite the frozen fixture. Compare disagreements manually.


## Summary

You planned evidence needs, built focused views over 28 heterogeneous sources, mapped them into validated records, exposed irrelevant and derivative evidence, classified four conflict types, filled one gap, preserved another, generated prose from a claim-evidence map, implemented structured Refine, measured order sensitivity, and evaluated the result.

**Key takeaway:** trustworthy synthesis exposes the evidence structure—including duplication, disagreement, and absence—before it produces polished prose.

Continue to [Intermediate 06 — Local Qdrant](../06-qdrant-local/README.md).
